# SafeStreet AI - Feature Engineering

## Objective
Prepare the cleaned dataset and create machine learning features for crime risk prediction.

In [1]:
import pandas as pd
import numpy as np

In [2]:
from pathlib import Path

DATA_PATH = Path("../data/raw/chicago_crime.csv")

df = pd.read_csv(DATA_PATH)

print(df.shape)
df.head()

(7846809, 22)


,ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,...,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
0,11646166,JC213529,09/01/2018 12:01:00 AM,082XX S INGLESIDE AVE,0810,THEFT,OVER $500,RESIDENCE,False,True,...,8.0,44.0,06,NaN,NaN,2018,04/06/2019 04:04:43 PM,NaN,NaN,NaN
1,11645836,JC212333,05/01/2016 12:25:00 AM,055XX S ROCKWELL ST,1153,DECEPTIVE PRACTICE,FINANCIAL IDENTITY THEFT OVER $ 300,NaN,False,False,...,15.0,63.0,11,NaN,NaN,2016,04/06/2019 04:04:43 PM,NaN,NaN,NaN
2,11449702,JB373031,07/31/2018 01:30:00 PM,009XX E HYDE PARK BLVD,2024,NARCOTICS,POSS: HEROIN(WHITE),STREET,True,False,...,5.0,41.0,18,NaN,NaN,2018,04/09/2019 04:24:58 PM,NaN,NaN,NaN
3,11643334,JC209972,12/19/2018 04:30:00 PM,056XX W WELLINGTON AVE,1320,CRIMINAL DAMAGE,TO VEHICLE,STREET,False,False,...,31.0,19.0,14,NaN,NaN,2018,04/04/2019 04:16:11 PM,NaN,NaN,NaN
4,11645527,JC212744,02/02/2015 10:00:00 AM,069XX W ARCHER AVE,1153,DECEPTIVE PRACTICE,FINANCIAL IDENTITY THEFT OVER $ 300,OTHER,False,False,...,23.0,56.0,11,NaN,NaN,2015,04/06/2019 04:04:43 PM,NaN,NaN,NaN


In [3]:
# Convert Date column to datetime

df["Date"] = pd.to_datetime(
    df["Date"],
    format="%m/%d/%Y %I:%M:%S %p"
)

print(df["Date"].dtype)

df[["Date"]].head()

datetime64[us]


,Date
0,2018-09-01 00:01:00
1,2016-05-01 00:25:00
2,2018-07-31 13:30:00
3,2018-12-19 16:30:00
4,2015-02-02 10:00:00


In [4]:
# Time-based features

df["Hour"] = df["Date"].dt.hour
df["DayOfWeek"] = df["Date"].dt.dayofweek
df["Month"] = df["Date"].dt.month
df["IsWeekend"] = df["DayOfWeek"].isin([5, 6]).astype(int)

df[[
    "Date",
    "Hour",
    "DayOfWeek",
    "Month",
    "IsWeekend"
]].head()

,Date,Hour,DayOfWeek,Month,IsWeekend
0,2018-09-01 00:01:00,0,5,9,1
1,2016-05-01 00:25:00,0,6,5,1
2,2018-07-31 13:30:00,13,1,7,0
3,2018-12-19 16:30:00,16,2,12,0
4,2015-02-02 10:00:00,10,0,2,0


In [5]:
# Remove rows without GPS coordinates

print("Original Shape :", df.shape)

df = df.dropna(subset=["Latitude", "Longitude"])

print("After GPS Cleaning :", df.shape)

Original Shape : (7846809, 26)
After GPS Cleaning : (7758698, 26)


In [6]:
df[["Latitude", "Longitude"]].isnull().sum()

Latitude     0
Longitude    0
dtype: int64

In [7]:
df = df.reset_index(drop=True)

In [8]:
severity_map = {
    "HOMICIDE": 5,
    "ROBBERY": 5,
    "CRIMINAL SEXUAL ASSAULT": 5,
    "CRIM SEXUAL ASSAULT": 5,
    "KIDNAPPING": 5,

    "BATTERY": 4,
    "ASSAULT": 4,
    "HUMAN TRAFFICKING": 4,

    "BURGLARY": 3,
    "MOTOR VEHICLE THEFT": 3,
    "WEAPONS VIOLATION": 3,
    "STALKING": 3,
    "INTIMIDATION": 3,

    "THEFT": 2,
    "CRIMINAL DAMAGE": 2,
    "CRIMINAL TRESPASS": 2,
    "OTHER OFFENSE": 2,
    "PUBLIC PEACE VIOLATION": 2,
    "OFFENSE INVOLVING CHILDREN": 2,
    "DECEPTIVE PRACTICE": 2,

    "LIQUOR LAW VIOLATION": 1,
    "GAMBLING": 1,
    "NARCOTICS": 1,
    "PROSTITUTION": 1,
    "NON-CRIMINAL": 1,
    "OBSCENITY": 1,
    "CONCEALED CARRY LICENSE VIOLATION": 1,
    "INTERFERENCE WITH PUBLIC OFFICER": 1,
    "SEX OFFENSE": 2,

    # Newly discovered crime types
    "ARSON": 4,
    "OTHER NARCOTIC VIOLATION": 1,
    "PUBLIC INDECENCY": 2,
    "NON - CRIMINAL": 1,
    "NON-CRIMINAL (SUBJECT SPECIFIED)": 1,
    "RITUALISM": 2,
    "DOMESTIC VIOLENCE": 4,
}

In [9]:
df["Severity"] = df["Primary Type"].map(severity_map)

df[["Primary Type", "Severity"]].head(10)

,Primary Type,Severity
0,CRIMINAL DAMAGE,2
1,PUBLIC PEACE VIOLATION,2
2,BATTERY,4
3,BATTERY,4
4,LIQUOR LAW VIOLATION,1
5,CRIMINAL DAMAGE,2
6,BURGLARY,3
7,THEFT,2
8,CRIMINAL TRESPASS,2
9,WEAPONS VIOLATION,3


In [10]:
missing_severity = df[df["Severity"].isna()]

print("Missing Severity Labels:", len(missing_severity))

missing_severity["Primary Type"].unique()

Missing Severity Labels: 0


<ArrowStringArray>
[]
Length: 0, dtype: str

In [11]:
def severity_to_risk(severity):
    if severity <= 2:
        return "Low"
    elif severity == 3:
        return "Medium"
    else:
        return "High"


df["RiskLevel"] = df["Severity"].apply(severity_to_risk)

df[["Severity", "RiskLevel"]].head(10)

,Severity,RiskLevel
0,2,Low
1,2,Low
2,4,High
3,4,High
4,1,Low
5,2,Low
6,3,Medium
7,2,Low
8,2,Low
9,3,Medium


In [12]:
risk_distribution = (
    df["RiskLevel"]
      .value_counts(normalize=True)
      .mul(100)
      .round(2)
)

print(risk_distribution)

RiskLevel
Low       58.53
High      29.61
Medium    11.87
Name: proportion, dtype: float64


In [13]:
print(df.shape)

(7758698, 28)


In [14]:
df["Severity"].value_counts().sort_index()

Severity
1     855473
2    3685419
3     920798
4    1951524
5     345484
Name: count, dtype: int64

In [15]:
df["Primary Type"].value_counts().loc[
    [
        "ARSON",
        "OTHER NARCOTIC VIOLATION",
        "PUBLIC INDECENCY",
        "NON - CRIMINAL",
        "NON-CRIMINAL (SUBJECT SPECIFIED)",
        "RITUALISM",
        "DOMESTIC VIOLENCE"
    ]
]

Primary Type
ARSON                               13297
OTHER NARCOTIC VIOLATION              145
PUBLIC INDECENCY                      196
NON - CRIMINAL                         38
NON-CRIMINAL (SUBJECT SPECIFIED)        9
RITUALISM                              23
DOMESTIC VIOLENCE                       1
Name: count, dtype: int64

# ==========================
# Feature Selection
# ==========================

In [16]:
features = [
    "Latitude",
    "Longitude",
    "Hour",
    "DayOfWeek",
    "Month",
    "IsWeekend",
    "District",
    "Community Area",
    "Location Description"
]

target = "RiskLevel"

In [17]:
X = df[features]
y = df[target]

print("Feature Matrix:", X.shape)
print("Target:", y.shape)

Feature Matrix: (7758698, 9)
Target: (7758698,)


In [18]:
X.isnull().sum()

Latitude                     0
Longitude                    0
Hour                         0
DayOfWeek                    0
Month                        0
IsWeekend                    0
District                    47
Community Area          604274
Location Description      6834
dtype: int64

In [19]:
pip install pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
pip show pyarrow

Name: pyarrow
Version: 24.0.0
Summary: Python library for Apache Arrow
Home-page: 
Author: 
Author-email: 
License: 
Location: D:\SOUMIT MANNA\IEM\Internship\Summer Internship\SafeStreet\ml_backend\.venv\Lib\site-packages
Requires: 
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [21]:
# ==========================================
# Create Final Training Dataset
# ==========================================

training_columns = [
    "Latitude",
    "Longitude",
    "Hour",
    "DayOfWeek",
    "Month",
    "IsWeekend",
    "District",
    "Community Area",
    "Location Description",
    "RiskLevel"
]

processed_df = df[training_columns].copy()

print("Training Dataset Shape:", processed_df.shape)

processed_df.head()

Training Dataset Shape: (7758698, 10)


,Latitude,Longitude,Hour,DayOfWeek,Month,IsWeekend,District,Community Area,Location Description,RiskLevel
0,41.813999,-87.598138,18,3,9,0,2.0,39.0,RESIDENCE,Low
1,41.954584,-87.648376,22,5,9,1,19.0,3.0,STREET,Low
2,41.808521,-87.626066,1,6,9,1,2.0,38.0,VACANT LOT/LAND,High
3,41.885759,-87.713588,2,6,9,1,11.0,27.0,SIDEWALK,High
4,41.947100,-87.662116,2,6,9,1,19.0,6.0,SIDEWALK,Low


In [22]:
from pathlib import Path

output_path = Path("../data/processed/processed_crime.parquet")

processed_df.to_parquet(
    output_path,
    index=False,
    compression="snappy"
)

print("Processed training dataset saved successfully!")


Processed training dataset saved successfully!


In [23]:
# ==========================================
# Create Development Sample
# ==========================================

from sklearn.model_selection import train_test_split

DEV_SAMPLE_SIZE = 1_000_000
RANDOM_STATE = 42

dev_df, _ = train_test_split(
    processed_df,
    train_size=DEV_SAMPLE_SIZE,
    stratify=processed_df["RiskLevel"],
    random_state=RANDOM_STATE
)

print("Development Dataset Shape:", dev_df.shape)

print()

print(
    dev_df["RiskLevel"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Development Dataset Shape: (1000000, 10)

RiskLevel
Low       58.53
High      29.61
Medium    11.87
Name: proportion, dtype: float64


In [24]:
# ==========================================
# Save Development Dataset
# ==========================================

from pathlib import Path

dev_output_path = Path("../data/processed/dev_sample.parquet")

dev_df.to_parquet(
    dev_output_path,
    index=False,
    compression="snappy"
)

print("Development dataset saved successfully!")
print(f"Saved at: {dev_output_path.resolve()}")

Development dataset saved successfully!
Saved at: D:\SOUMIT MANNA\IEM\Internship\Summer Internship\SafeStreet\ml_backend\data\processed\dev_sample.parquet


# ==========================================
# Feature Engineering V2
# ==========================================

## V2.1 Time-Based Features

In [25]:
# ==========================================
# TimeOfDay Feature
# ==========================================

def get_time_of_day(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

processed_df["TimeOfDay"] = processed_df["Hour"].apply(get_time_of_day)

print("TimeOfDay Feature Created Successfully!")

processed_df["TimeOfDay"].value_counts()

TimeOfDay Feature Created Successfully!


TimeOfDay
Night        2375001
Afternoon    2008758
Evening      1700454
Morning      1674485
Name: count, dtype: int64

In [26]:
# ==========================================
# IsNight Feature
# ==========================================

processed_df["IsNight"] = (
    (processed_df["Hour"] >= 21) |
    (processed_df["Hour"] < 5)
).astype(int)

print("IsNight Feature Created Successfully!\n")

print(processed_df["IsNight"].value_counts())

print("\nPercentage Distribution:")

print(
    processed_df["IsNight"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

IsNight Feature Created Successfully!

IsNight
0    5383697
1    2375001
Name: count, dtype: int64

Percentage Distribution:
IsNight
0    69.39
1    30.61
Name: proportion, dtype: float64


In [27]:
# ==========================================
# IsBusinessHours Feature
# ==========================================

processed_df["IsBusinessHours"] = (
    (processed_df["Hour"] >= 9) &
    (processed_df["Hour"] < 18)
).astype(int)

print("IsBusinessHours Feature Created Successfully!\n")

print(processed_df["IsBusinessHours"].value_counts())

print("\nPercentage Distribution:")

print(
    processed_df["IsBusinessHours"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

IsBusinessHours Feature Created Successfully!

IsBusinessHours
0    4346665
1    3412033
Name: count, dtype: int64

Percentage Distribution:
IsBusinessHours
0    56.02
1    43.98
Name: proportion, dtype: float64


In [28]:
# ==========================================
# Cyclical Encoding of Hour
# ==========================================

import numpy as np

processed_df["Hour_sin"] = np.sin(
    2 * np.pi * processed_df["Hour"] / 24
)

processed_df["Hour_cos"] = np.cos(
    2 * np.pi * processed_df["Hour"] / 24
)

print("Hour cyclical features created successfully!\n")

processed_df[
    ["Hour", "Hour_sin", "Hour_cos"]
].head(10)

Hour cyclical features created successfully!



,Hour,Hour_sin,Hour_cos
0,18,-1.000000,-1.836970e-16
1,22,-0.500000,8.660254e-01
2,1,0.258819,9.659258e-01
3,2,0.500000,8.660254e-01
4,2,0.500000,8.660254e-01
5,0,0.000000,1.000000e+00
6,20,-0.866025,5.000000e-01
7,20,-0.866025,5.000000e-01
8,21,-0.707107,7.071068e-01
9,0,0.000000,1.000000e+00


In [29]:
# ==========================================
# Cyclical Encoding of Month
# ==========================================

processed_df["Month_sin"] = np.sin(
    2 * np.pi * processed_df["Month"] / 12
)

processed_df["Month_cos"] = np.cos(
    2 * np.pi * processed_df["Month"] / 12
)

print("Month cyclical features created successfully!\n")

processed_df[
    ["Month", "Month_sin", "Month_cos"]
].head(10)

Month cyclical features created successfully!



,Month,Month_sin,Month_cos
0,9,-1.0,-1.836970e-16
1,9,-1.0,-1.836970e-16
2,9,-1.0,-1.836970e-16
3,9,-1.0,-1.836970e-16
4,9,-1.0,-1.836970e-16
5,9,-1.0,-1.836970e-16
6,9,-1.0,-1.836970e-16
7,9,-1.0,-1.836970e-16
8,9,-1.0,-1.836970e-16
9,9,-1.0,-1.836970e-16


# ==========================================
# Save Feature Engineering V2 Dataset
# ==========================================

In [30]:
from pathlib import Path

# ==========================================
# Save Feature Engineering V2 Dataset
# ==========================================

output_path = Path("../data/processed/processed_crime_v2.parquet")

processed_df.to_parquet(
    output_path,
    index=False,
    compression="snappy"
)

print("Feature Engineering V2 dataset saved successfully!")
print(output_path.resolve())

Feature Engineering V2 dataset saved successfully!
D:\SOUMIT MANNA\IEM\Internship\Summer Internship\SafeStreet\ml_backend\data\processed\processed_crime_v2.parquet
